In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType ,IntegerType,DataType ,TimestampType,FloatType
catalog_name='ecommers'

# bronze_brands

In [0]:
df_bronze=spark.table(f'{catalog_name}.bronze.brz_brands')

In [0]:
df_bronze.show(10)

In [0]:
display(df_bronze.schema)

In [0]:
df_bronze.printSchema()

In [0]:
df_silver=df_bronze.withColumn("brand_name",F.trim(F.col("brand_name")))

In [0]:
df_silver.show(5)

In [0]:
df_silver = df_silver.withColumn("brand_code", F.regexp_replace(F.col("brand_code"), "[^a-zA-Z0-9]", ""))

In [0]:
df_silver.select("category_code").distinct().show()

In [0]:
# Anomalies dictionary
anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS": "TOY"
}

# PySpark replace is easy
df_silver = df_silver.replace(
    to_replace=anomalies,
    subset=["category_code"]
)

# ✅ Show results
display(df_silver.select("category_code").distinct().show())

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_brands)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

# Category

In [0]:
df=spark.table(f"{catalog_name}.bronze.brz_category")

In [0]:
display(df)

In [0]:
# Convert category_code to uppercase, trim spaces from category_code and category_name
df = df.withColumn("category_code", F.upper(F.trim(F.col("category_code")))) \
       .withColumn("category_name", F.trim(F.col("category_name")))

# Show distinct values of category_code and category_name
df.select("category_code").distinct().show(truncate=False)
df.select("category_name").distinct().show(truncate=False)

In [0]:
# Write cleaned data to the silver layer
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_category")

#Products

In [0]:
df=spark.table(f"{catalog_name}.bronze.brz_products")

In [0]:
# Convert category_code and brand_code to uppercase
df = df.withColumn("category_code", F.upper(F.col("category_code")))
df = df.withColumn("brand_code", F.upper(F.col("brand_code")))

# Drop rows where category_code is null
df = df.dropna(subset=["category_code"])

# Fix size column - trim spaces and standardize, and normalize to 'ONE SIZE'
df = df.withColumn("size", F.trim(F.col("size")))
df = df.withColumn("size", F.when(F.upper(F.col("size")) == "ONE-SIZE", "OS").otherwise(F.col("size")))

# Remove 'g' from weight_grams and cast to float
df = df.withColumn("weight_grams", F.regexp_replace(F.col("weight_grams"), "[gG]", "").cast(FloatType()))

display(df)

In [0]:
# Find all distinct values for key columns and check for anomalies
print("Distinct category_code values:")
df.select("category_code").distinct().show(truncate=False)

print("Distinct brand_code values:")
df.select("brand_code").distinct().show(truncate=False)

print("Distinct size values:")
df.select("size").distinct().show(truncate=False)

print("Distinct weight_grams values:")
df.select("weight_grams").distinct().orderBy("weight_grams").show(truncate=False)

# Check for nulls in each column
from pyspark.sql.functions import col, sum as _sum
df.select([_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

In [0]:
# Fill null values in the 'color' column with 'Unknown'
df = df.withColumn("color", F.when(F.col("color").isNull(), "Unknown").otherwise(F.col("color")))

In [0]:
from pyspark.sql.functions import col, sum as _sum
df.select([_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

In [0]:
# Write cleaned data to the silver layer
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_products")

#products

In [0]:
df=spark.table(f"{catalog_name}.bronze.brz_customers")

In [0]:
display(df)

# Check for nulls in each column
from pyspark.sql.functions import col, sum as _sum
df.select([_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

In [0]:
# Remove rows where customer_id is null
df = df.dropna(subset=["customer_id"])

# Fill null values in string columns with 'Unknown'
string_cols = [c for c, dtype in df.dtypes if dtype == "string" and c != "customer_id"]
for col_name in string_cols:
    df = df.withColumn(col_name, F.when(F.col(col_name).isNull(), "Unknown").otherwise(F.col(col_name)))

# Verify nulls are handled
from pyspark.sql.functions import col, sum as _sum
df.select([_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

display(df)

In [0]:
# Show distinct values for key columns
print("Distinct customer_id values:")
df.select("customer_id").distinct().show(truncate=False)

for col_name in df.columns:
    print(f"Distinct {col_name} values: {df.select(col_name).distinct().count()}")
    df.select(col_name).distinct().show(truncate=False)

# Calendar

In [0]:
df = spark.table(f"{catalog_name}.bronze.brz_calendar")

# Show distinct values for each column
for col_name in df.columns:
    print(f"Distinct {col_name} values: {df.select(col_name).distinct().count()}")
    df.select(col_name).distinct().show(truncate=False)

display(df)

In [0]:
# Clean calendar data: uppercase day_name and ensure week_of_year is non-negative
df = df.withColumn("day_name", F.upper(F.col("day_name")))
df = df.withColumn("week_of_year", F.abs(F.col("week_of_year")))

# Validate week_of_year against the date column
df = df.withColumn("expected_week_of_year", F.weekofyear(F.col("date")))
df = df.withColumn("week_of_year_mismatch", F.col("week_of_year") != F.col("expected_week_of_year"))

# Show mismatches if any
print("Rows where week_of_year does not match the date:")
df.filter(F.col("week_of_year_mismatch") == True).show(truncate=False)

display(df)

In [0]:
# Remove duplicates from df
df = df.dropDuplicates()

In [0]:
# Write cleaned data to the silver layer
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_calendar")